In [3]:
import json
import warnings
import numpy as np
import mne
from pathlib import Path

BASE_DIR  = Path("/home/gmarihuan/chbmit")
JSON_PATH = BASE_DIR / "chbmit_summary.json"

# MNE's get_data() returns volts (~1e-5). The network's BatchNorm uses
# eps=1e-5, so volts-scale inputs sit below eps and never normalize -> the
# loss locks at ln(2). Converting to microvolts (x1e6) puts the signal at
# ~tens of uV, well above eps, and preserves cross-window amplitude (the
# ictal vs. inter-ictal amplitude difference). No data statistics are used,
# so this introduces no train/test leakage.
VOLTS_TO_MICROVOLTS = 1e6

REQUIRED_CHANNELS = [
    "FP1-F7", "F7-T7", "P7-O1",
    "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
    "FP2-F4", "F4-C4", "C4-P4", "P4-O2",
    "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
    "FZ-CZ", "CZ-PZ",
    "P7-T7", "T7-FT9", "FT9-FT10", "FT10-T8",
]


def _load_summary() -> dict:
    with open(JSON_PATH) as f:
        return json.load(f)


def _read_required(filepath: Path, lowpass_hz: float | None = 64.0) -> mne.io.BaseRaw:
    """Read an EDF, collapse the duplicate T8-P8 name, keep+reorder required
    channels, and (optionally) apply the paper's lowpass filter.

    lowpass_hz : float or None
        Cutoff for the lowpass filter in Hz. The paper (Section III-A-1)
        filters out noise above 64 Hz. Pass None to skip filtering (e.g. to
        isolate the effect of the microvolt rescaling).
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Channel names are not unique",
            category=RuntimeWarning,
        )
        raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
    if "T8-P8-0" in raw.ch_names:
        raw.rename_channels({"T8-P8-0": "T8-P8"})
    raw.pick(REQUIRED_CHANNELS)   # selects + reorders to match REQUIRED_CHANNELS

    # Paper preprocessing: lowpass < 64 Hz. Done on the full recording before
    # segmentation so the filter has the whole signal to work with.
    if lowpass_hz is not None:
        raw.filter(l_freq=None, h_freq=lowpass_hz, verbose=False)
    return raw


def _channel_names(filepath: Path) -> list[str]:
    """Return channel names from an EDF file (after collapsing the duplicate)."""
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Channel names are not unique",
            category=RuntimeWarning,
        )
        raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
    if "T8-P8-0" in raw.ch_names:
        raw.rename_channels({"T8-P8-0": "T8-P8"})
    return raw.ch_names


def _norm(ch: str) -> str:
    # collapse en-dash (–) and em-dash (—) to a plain hyphen, then strip/upper
    return ch.strip().upper().replace("\u2013", "-").replace("\u2014", "-")


def filter_files_with_channels(file_list, required=REQUIRED_CHANNELS):
    required_set = {_norm(c) for c in required}
    kept = []
    for f in file_list:
        channels = {_norm(c) for c in _channel_names(f)}
        if required_set.issubset(channels):
            kept.append(f)
    return kept


def _extract_seizure_times(seizure_info: dict):
    """
    Pull seizure start/end times (in seconds) out of a summary file-entry.

    Expected JSON shape:
        "chb01_03.edf": {
            ...,
            "seizures": {"1": {"start_sec": 2996, "end_sec": 3036}, ...}
        }
    """
    starts, ends = [], []
    if not seizure_info:
        return starts, ends

    seizures = seizure_info.get("seizures", {})
    for key in sorted(seizures, key=lambda k: int(k)):
        ev = seizures[key]
        starts.append(float(ev["start_sec"]))
        ends.append(float(ev["end_sec"]))

    return starts, ends


def split_seizure_segments(data, sfreq, seizure_starts, seizure_ends):
    """
    Split already-loaded EEG data into seizure and normal segments by time.

    Returns
    -------
    seizure_segments : list[np.ndarray]   one (n_channels, n_times) per seizure
    normal_segments  : list[np.ndarray]   gaps before/between/after seizures
    """
    if len(seizure_starts) != len(seizure_ends):
        raise ValueError("seizure_starts and seizure_ends must be the same length.")

    n_times = data.shape[1]

    intervals = sorted(
        (int(round(s * sfreq)), int(round(e * sfreq)))
        for s, e in zip(seizure_starts, seizure_ends)
    )
    intervals = [(max(0, s), min(n_times, e)) for s, e in intervals]

    seizure_segments, normal_segments = [], []
    cursor = 0
    for s, e in intervals:
        if s > cursor:
            normal_segments.append(data[:, cursor:s])
        seizure_segments.append(data[:, s:e])
        cursor = e
    if cursor < n_times:
        normal_segments.append(data[:, cursor:n_times])

    return seizure_segments, normal_segments


def load_patient_data(patient_name: str, base_dir: str = str(BASE_DIR),
                      normal_from_seizure_only: bool = False,
                      lowpass_hz: float | None = 64.0,
                      to_microvolts: bool = True):
    """
    Load and label all EEG data for one patient as per-segment flattened vectors.

    Parameters
    ----------
    normal_from_seizure_only : bool, default False
        If True, drop every fully-normal (non-seizure) recording and keep only
        the segments carved out of seizure recordings. The seizure (label 1)
        segments ARE still returned; only the standalone normal recordings are
        discarded. Result: a mix of label 0 (inter-ictal gaps) and label 1
        (seizure) windows, with the majority-class normal recordings removed.
    lowpass_hz : float or None, default 64.0
        Lowpass cutoff in Hz applied per recording (paper Section III-A-1).
        Pass None to skip filtering.
    to_microvolts : bool, default True
        Multiply the volts-scale data from MNE by 1e6 -> microvolts. Required
        for the model to train (see VOLTS_TO_MICROVOLTS note above). Set False
        only if your raw data is already in non-volt units.

    Returns
    -------
    X : np.ndarray, shape (n_segments,), dtype=object
    y : np.ndarray, shape (n_segments,), dtype int8
    """
    summary         = _load_summary()
    patient_summary = summary.get(patient_name, {})

    patient_dir = Path(base_dir) / patient_name
    if not patient_dir.is_dir():
        raise FileNotFoundError(f"Patient directory not found: {patient_dir}")

    all_edfs = sorted(patient_dir.glob("*.edf"))

    def _has_seizures(name: str) -> bool:
        info = patient_summary.get(name)
        return bool(info and info.get("seizures"))

    seizure_edfs = [f for f in all_edfs if _has_seizures(f.name)]
    normal_edfs  = [f for f in all_edfs if not _has_seizures(f.name)]

    seizure_edfs = filter_files_with_channels(seizure_edfs, REQUIRED_CHANNELS)
    normal_edfs  = filter_files_with_channels(normal_edfs,  REQUIRED_CHANNELS)

    scale = VOLTS_TO_MICROVOLTS if to_microvolts else 1.0

    X_segments: list[np.ndarray] = []
    y_labels:   list[int]        = []

    for filepath in seizure_edfs:
        raw   = _read_required(filepath, lowpass_hz=lowpass_hz)
        data  = raw.get_data() * scale          # (n_channels, n_times), microvolts
        sfreq = raw.info["sfreq"]

        starts, ends = _extract_seizure_times(patient_summary.get(filepath.name))
        seiz_segs, norm_segs = split_seizure_segments(data, sfreq, starts, ends)

        for seg in seiz_segs:
            X_segments.append(seg.reshape(-1))   # flatten (channels × time)
            y_labels.append(1)

        for seg in norm_segs:
            X_segments.append(seg.reshape(-1))
            y_labels.append(0)

    # Skip the fully-normal recordings entirely when we only want the
    # inter-ictal segments carved out of seizure files.
    if not normal_from_seizure_only:
        for filepath in normal_edfs:
            raw  = _read_required(filepath, lowpass_hz=lowpass_hz)
            data = raw.get_data() * scale            # one normal segment per file
            X_segments.append(data.reshape(-1))
            y_labels.append(0)

    if not X_segments:
        raise ValueError(
            f"No segments produced for patient '{patient_name}' "
            f"(check required channels / normal_from_seizure_only setting)."
        )

    # Object array because segments have different lengths.
    X = np.empty(len(X_segments), dtype=object)
    for i, seg in enumerate(X_segments):
        X[i] = seg
    y = np.asarray(y_labels, dtype=np.int8)

    return X, y

In [4]:
import numpy as np

N_CHANNELS = len(REQUIRED_CHANNELS)   # 21


def segments_to_windows(
    X_segments: np.ndarray,
    y_segments: np.ndarray,
    window_samples: int,
    overlap_sec: float = 0.0,
    sfreq: float = 256.0,
    n_channels: int = N_CHANNELS,
    overlap_seizure_only: bool = False,
    seizure_label: int = 1,
):
    """
    Turn variable-length segments into a fixed-size 3D array of windows.

    Parameters
    ----------
    X_segments : np.ndarray, shape (n_segments,), dtype=object
        Output of load_patient_data — each element is a 1D vector of length
        (n_channels * n_times_i), channel-major (channel 0's samples first).
    y_segments : np.ndarray, shape (n_segments,)
        Per-segment labels (1 = seizure, 0 = normal).
    window_samples : int
        Window length in TIME samples (e.g. 256*4 = 1024 for 4 s at 256 Hz).
    overlap_sec : float
        Overlap between consecutive windows, in SECONDS. 0.0 = no overlap.
        Must be less than the window length in seconds.
    sfreq : float
        Sampling frequency in Hz, used to convert overlap_sec to samples.
    n_channels : int
        Channels per segment, needed to un-flatten. Defaults to len(REQUIRED_CHANNELS).
    overlap_seizure_only : bool
        If True, apply `overlap_sec` only to seizure segments; normal segments
        are windowed with no overlap. Useful to up-sample the minority (seizure)
        class without inflating the majority class. Default False (overlap
        applied to all segments).
    seizure_label : int
        Label value that marks a seizure segment. Default 1.

    Returns
    -------
    X : np.ndarray, shape (n_windows, n_channels, window_samples), float
    y : np.ndarray, shape (n_windows,), int8
        Each window inherits the label of the segment it came from.
    """
    if window_samples <= 0:
        raise ValueError("window_samples must be positive.")

    overlap_samples = int(round(overlap_sec * sfreq))
    if overlap_samples < 0:
        raise ValueError("overlap_sec must be non-negative.")
    if overlap_samples >= window_samples:
        raise ValueError(
            f"overlap ({overlap_samples} samples) must be smaller than "
            f"window_samples ({window_samples})."
        )

    # hop between window starts: with overlap vs. without
    overlap_step = window_samples - overlap_samples
    no_overlap_step = window_samples

    X_windows, y_windows = [], []

    for flat, label in zip(X_segments, y_segments):
        # un-flatten (channels × time) → (n_channels, n_times)
        seg = np.asarray(flat).reshape(n_channels, -1)
        n_times = seg.shape[1]

        # too short for even one window → skip
        if n_times < window_samples:
            continue

        # pick the hop for THIS segment
        if overlap_seizure_only and label != seizure_label:
            step = no_overlap_step       # normal segment: no overlap
        else:
            step = overlap_step          # seizure (or overlap-for-all): use overlap

        for start in range(0, n_times - window_samples + 1, step):
            X_windows.append(seg[:, start:start + window_samples])
            y_windows.append(label)

    if not X_windows:
        raise ValueError(
            "No windows produced — every segment was shorter than "
            f"window_samples ({window_samples})."
        )

    X = np.stack(X_windows, axis=0).astype(np.float64)   # (n_windows, n_channels, window)
    y = np.asarray(y_windows, dtype=np.int8)
    return X, y

# Model

In [5]:
"""
EEGWaveNet — subject-dependent reproduction (CHB-MIT).

Reconstructed from:
Thuwajit et al., "EEGWaveNet: Multiscale CNN-Based Spatiotemporal Feature
Extraction for EEG Seizure Detection," IEEE Trans. Ind. Informat., 18(8), 2022.

Architecture source: Section III-B and Table I.

Two points where the paper's text is internally inconsistent and a choice had
to be made (both flagged inline):

  1. PADDING. Section III-B-2 says the spatial-temporal convs use "no padding."
     With kernel 4 / stride 2 and the stated scales, the smallest branch
     (length 16) reduces to length 0 on the 3rd conv, so the network cannot be
     built as literally described. padding=1 makes every branch valid and
     matches the authors' released implementation. Exposed as `st_padding`
     (default 1). Set st_padding=0 to reproduce the literal text (will crash on
     the 16-length branch with T=1024).

  2. OUTPUT / LOSS. Section III-B-3 says the last layer is Log-Softmax, while
     Section III-C says the loss is cross-entropy. Log-Softmax pairs with
     NLLLoss, not CrossEntropyLoss (which applies log-softmax internally).
     `return_log_softmax=True` (default) returns log-probabilities -> use
     nn.NLLLoss. Set False to return raw logits -> use nn.CrossEntropyLoss.
     The two combinations are mathematically equivalent; pick one, not both.

Input tensor shape: (N, C, T) = (batch, 21, 1024). The paper's (C, 1, T) layout
is implemented with Conv1d (the singleton height dim is dropped).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class _SpatialTemporalBranch(nn.Module):
    """Three regular conv1d layers (kernel 4, stride 2), each + BN + LeakyReLU,
    followed by global average pooling over time -> 32 features.
    'Regular' (groups=1) so it mixes across all EEG channels (Section III-B-2)."""

    def __init__(self, in_channels: int, feat: int = 32, padding: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, feat, kernel_size=4, stride=2, padding=padding),
            nn.BatchNorm1d(feat),
            nn.LeakyReLU(),
            nn.Conv1d(feat, feat, kernel_size=4, stride=2, padding=padding),
            nn.BatchNorm1d(feat),
            nn.LeakyReLU(),
            nn.Conv1d(feat, feat, kernel_size=4, stride=2, padding=padding),
            nn.BatchNorm1d(feat),
            nn.LeakyReLU(),
        )

    def forward(self, x):                  # x: (N, in_channels, L)
        x = self.block(x)                  # (N, 32, L')
        return x.mean(dim=-1)              # global avg pool -> (N, 32)


class EEGWaveNet(nn.Module):
    """EEGWaveNet for binary seizure / non-seizure detection.

    Modules (Section III-B):
      A. Multiscale convolution: 6 cascaded depthwise convs (kernel 2, stride 2),
         each halving the temporal resolution. Outputs of layers 2-6 (5 scales)
         feed the spatial-temporal module.
      B. Spatial-temporal extraction: one independent branch per scale, each
         producing 32 features -> 5 x 32 = 160 features.
      C. Classifier: FC 160 -> 64 -> 32 -> n_classes.
    """

    def __init__(
        self,
        n_channels: int = 21,
        n_classes: int = 2,
        feat_per_scale: int = 32,
        st_padding: int = 1,          # see note 1 in module docstring
        return_log_softmax: bool = True,  # see note 2 in module docstring
    ):
        super().__init__()
        self.return_log_softmax = return_log_softmax

        # --- Module A: multiscale depthwise convolution (6 layers) ---
        # groups=n_channels => depthwise (no info crossing between channels).
        self.multiscale = nn.ModuleList([
            nn.Conv1d(n_channels, n_channels, kernel_size=2, stride=2,
                      groups=n_channels)
            for _ in range(6)
        ])

        # --- Module B: 5 spatial-temporal branches (for layer-2..6 outputs) ---
        self.branches = nn.ModuleList([
            _SpatialTemporalBranch(n_channels, feat_per_scale, st_padding)
            for _ in range(5)
        ])

        # --- Module C: classifier (Fig. 1c: 160 -> 64 -> 32 -> n_classes) ---
        self.classifier = nn.Sequential(
            nn.Linear(5 * feat_per_scale, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 32),
            nn.LeakyReLU(),
            nn.Linear(32, n_classes),
        )

    def forward(self, x):                  # x: (N, C, T)
        # Cascade through the 6 depthwise layers, keeping outputs 2..6.
        outs = []
        for i, layer in enumerate(self.multiscale):
            x = layer(x)
            if i >= 1:                     # layers 2..6 (0-indexed 1..5)
                outs.append(x)
        assert len(outs) == 5

        feats = [branch(o) for branch, o in zip(self.branches, outs)]
        feats = torch.cat(feats, dim=1)    # (N, 160)

        logits = self.classifier(feats)    # (N, n_classes)
        if self.return_log_softmax:
            return F.log_softmax(logits, dim=1)   # pair with nn.NLLLoss
        return logits                              # pair with nn.CrossEntropyLoss


if __name__ == "__main__":
    # Shape sanity check (T=1024 == 4 s @ 256 Hz, the paper's setup).
    model = EEGWaveNet(n_channels=21, n_classes=2)
    n_params = sum(p.numel() for p in model.parameters())
    dummy = torch.randn(8, 21, 1024)
    out = model(dummy)
    print(f"params: {n_params:,}")
    print(f"input : {tuple(dummy.shape)}")
    print(f"output: {tuple(out.shape)}  (log-probabilities)")

params: 68,668
input : (8, 21, 1024)
output: (8, 2)  (log-probabilities)


In [6]:
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class EEGWindowDataset(Dataset):
    """Wraps windowed EEG (N, C, T) and labels (N,) as tensors."""

    def __init__(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y).reshape(-1)
        if X.ndim != 3:
            raise ValueError(
                f"X must be (N, C, T); got {X.shape}. If your windows are "
                f"(N, T, C), transpose with X.transpose(0, 2, 1)."
            )
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"N mismatch: X has {X.shape[0]}, y has {y.shape[0]}.")
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.y[i]


def make_loader(X, y, batch_size=512, shuffle=False, num_workers=0):
    """Build a DataLoader for windowed EEG arrays."""
    ds = EEGWindowDataset(X, y)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


def compute_class_weights(y, n_classes=2):
    """Inverse-frequency ('balanced') class weights: N / (n_classes * count_c).
    Returns a float tensor of length n_classes. Note: the paper applies class
    weights but does not state the exact formula."""
    y = np.asarray(y).reshape(-1)
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    counts[counts == 0] = 1.0  # guard against empty class in a fold
    weights = y.shape[0] / (n_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)


# --------------------------------------------------------------------------- #
# Metrics (binary; positive class = seizure = 1)
# --------------------------------------------------------------------------- #
def binary_metrics(y_true, y_pred):
    """Returns the paper's metrics from raw 0/1 predictions."""
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))

    def safe_div(a, b):
        return a / b if b else 0.0

    accuracy = safe_div(tp + tn, tp + tn + fp + fn)
    sensitivity = safe_div(tp, tp + fn)                 # recall, positive class
    specificity = safe_div(tn, tn + fp)
    f1_pos = safe_div(2 * tp, 2 * tp + fp + fn)         # binary F1 (seizure)
    f1_neg = safe_div(2 * tn, 2 * tn + fn + fp)         # F1 for non-seizure
    n_pos, n_neg = tp + fn, tn + fp
    f1_weighted = safe_div(f1_pos * n_pos + f1_neg * n_neg, n_pos + n_neg)

    return {
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1_binary": f1_pos,
        "f1_weighted": f1_weighted,
    }


# --------------------------------------------------------------------------- #
# Trainer
# --------------------------------------------------------------------------- #
@dataclass
class TrainConfig:
    lr: float = 1e-3
    batch_size: int = 512
    max_epochs: int = 1000          # cap; early stopping normally ends sooner
    patience: int = 10              # epochs without val-loss improvement
    n_classes: int = 2
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")
    verbose: bool = True


class Trainer:
    """Single train/val fit with class-weighted loss and early stopping,
    then evaluation on a test loader. Drive this from your LORO/5-fold loop."""

    def __init__(self, model: nn.Module, criterion: Optional[nn.Module],
                optimizer: Optional[torch.optim.Optimizer],
                config: TrainConfig = TrainConfig()):
        self.cfg = config
        self.device = torch.device(config.device)
        self.model = model.to(self.device)

        self.criterion = criterion
        self.optimizer = optimizer

    # -- internal: one pass over a loader -------------------------------------
    def _run_epoch(self, loader, train: bool):
        self.model.train(train)
        total_loss, n = 0.0, 0
        preds, trues = [], []
        torch.set_grad_enabled(train)
        for xb, yb in loader:
            xb, yb = xb.to(self.device), yb.to(self.device)
            out = self.model(xb)                     # (B, n_classes) log-probs
            loss = self.criterion(out, yb)
            if train:
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
            bs = yb.size(0)
            total_loss += loss.item() * bs
            n += bs
            preds.append(out.argmax(1).detach().cpu().numpy())
            trues.append(yb.detach().cpu().numpy())
        torch.set_grad_enabled(True)
        avg_loss = total_loss / max(n, 1)
        metrics = binary_metrics(np.concatenate(trues), np.concatenate(preds))
        return avg_loss, metrics

    # -- public API -----------------------------------------------------------
    def fit(self, train_loader, val_loader):
        """Train with early stopping; restores best (min val-loss) weights."""
        best_val = float("inf")
        best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
        stale = 0
        history = []

        for epoch in range(1, self.cfg.max_epochs + 1):
            tr_loss, tr_m = self._run_epoch(train_loader, train=True)
            va_loss, va_m = self._run_epoch(val_loader, train=False)
            history.append({"epoch": epoch, "train_loss": tr_loss,
                            "val_loss": va_loss, **{f"val_{k}": v for k, v in va_m.items()}})

            improved = va_loss < best_val
            if improved:
                best_val = va_loss
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
                stale = 0
            else:
                stale += 1

            if self.cfg.verbose:
                print(f"epoch {epoch:3d} | train {tr_loss:.4f} | val {va_loss:.4f} "
                      f"| val_f1b {va_m['f1_binary']:.4f} | sens {va_m['sensitivity']:.4f}"
                      f"{'  *best' if improved else ''}")

            if stale >= self.cfg.patience:
                if self.cfg.verbose:
                    print(f"early stopping at epoch {epoch} (no val improvement "
                          f"for {self.cfg.patience} epochs)")
                break

        self.model.load_state_dict(best_state)        # restore best weights
        return history

    @torch.no_grad()
    def evaluate(self, loader):
        """Return metrics dict on a loader using current (best) weights."""
        _, metrics = self._run_epoch(loader, train=False)
        return metrics

In [7]:
import numpy as np
import matplotlib.pyplot as plt

def plot_eeg(data, fs=256, ch_names=None, spacing=None,
             color="black", linewidth=0.5, figsize=(12, None), title="EEG"):
    """
    Plot multichannel EEG as vertically stacked traces.

    Parameters
    ----------
    data : array, shape (C, T)   -- C channels, T time samples
    fs : float                   -- sampling rate (Hz), sets the time axis
    ch_names : list[str] | None  -- channel labels (len C); defaults to ch0..
    spacing : float | None       -- vertical gap between traces in data units;
                                    auto-estimated from the data if None
    """
    data = np.asarray(data)
    if data.ndim != 2:
        raise ValueError(f"expected (C, T), got {data.shape}")
    C, T = data.shape

    if ch_names is None:
        ch_names = [f"ch{i}" for i in range(C)]

    t = np.arange(T) / fs

    # Auto spacing: a few times the median per-channel std keeps traces apart.
    if spacing is None:
        spacing = 4.0 * np.median(np.std(data, axis=1)) + 1e-9

    # Channel 0 at top, last channel at bottom (EEG montage convention).
    offsets = np.arange(C)[::-1] * spacing

    if figsize[1] is None:
        figsize = (figsize[0], max(2.0, 0.4 * C))

    fig, ax = plt.subplots(figsize=figsize)
    for i in range(C):
        ax.plot(t, data[i] + offsets[i], color=color, linewidth=linewidth)

    ax.set_yticks(offsets)
    ax.set_yticklabels(ch_names)
    ax.set_xlim(t[0], t[-1])
    ax.set_xlabel("Time (s)")
    ax.set_title(title)
    ax.margins(y=0.02)
    fig.tight_layout()
    return fig, ax

In [8]:
import numpy as np


def balance_dataset(X, y, method="undersample", seed=0):
    """
    Balance a windowed dataset so both classes have equal counts.

    Parameters
    ----------
    X : np.ndarray, shape (n_windows, n_channels, window_samples)
    y : np.ndarray, shape (n_windows,)   binary labels (0/1)
    method : {"undersample", "oversample"}
        "undersample" — randomly drop majority windows down to the minority
                        count (no duplicates, but discards data).
        "oversample"  — randomly duplicate minority windows up to the majority
                        count (keeps all data, but repeats minority windows).
    seed : int
        RNG seed for reproducible sampling.

    Returns
    -------
    X_bal, y_bal : balanced arrays, shuffled, same dtypes as inputs.
    """
    y = np.asarray(y)
    rng = np.random.default_rng(seed)

    classes, counts = np.unique(y, return_counts=True)
    if len(classes) != 2:
        raise ValueError(f"Expected 2 classes, found {classes.tolist()}.")

    target = counts.min() if method == "undersample" else counts.max()
    if method not in ("undersample", "oversample"):
        raise ValueError("method must be 'undersample' or 'oversample'.")

    keep_idx = []
    for cls in classes:
        cls_idx = np.flatnonzero(y == cls)
        # replace=True only matters for oversampling a minority class
        replace = method == "oversample" and len(cls_idx) < target
        chosen = rng.choice(cls_idx, size=target, replace=replace)
        keep_idx.append(chosen)

    keep_idx = rng.permutation(np.concatenate(keep_idx))   # shuffle classes together
    return X[keep_idx], y[keep_idx]

In [18]:
X_seg, y_seg = load_patient_data("chb01", normal_from_seizure_only=True)          # variable-length segments

# 4-second windows at 256 Hz, no overlap
X, y = segments_to_windows(X_seg, y_seg, window_samples=256 * 4, overlap_sec=0.0, overlap_seizure_only=True)

print(X.shape, y.shape)
print(int(y.sum()), "seizure windows of", len(y))

vals, cnts = np.unique(y, return_counts=True)
print({int(v): int(c) for v, c in zip(vals, cnts)})
print("positive fraction:", (y == 1).mean())   # ~0.2

n_neg = (y == 0).sum()
n_pos = (y == 1).sum()

minority = min(n_neg, n_pos)
print(f"{n_neg/minority:.0f}:{n_pos/minority:.0f}  (neg:pos)")

(5974, 21, 1024) (5974,)
108 seizure windows of 5974
{0: 5866, 1: 108}
positive fraction: 0.018078339471041177
54:1  (neg:pos)


In [ ]:
#X_bal, y_bal = balance_dataset(X, y, method="undersample", seed=0)

#vals, cnts = np.unique(y_bal, return_counts=True)
#print({int(v): int(c) for v, c in zip(vals, cnts)})   # {0: n_pos, 1: n_pos}
#print("positive fraction:", (y_bal == 1).mean())      # 0.5

{0: 108, 1: 108}
positive fraction: 0.5


In [ ]:
from sklearn.model_selection import train_test_split

# First peel off the test set (100/600 ≈ 0.167), stratified on y.
X_trval, X_te, y_trval, y_te = train_test_split(
    X, y, test_size=100, stratify=y, random_state=0
)
# Then split the remainder into train/val (100/500 = 0.2).
X_tr, X_va, y_tr, y_va = train_test_split(
    X_trval, y_trval, test_size=100, stratify=y_trval, random_state=0
)

cfg = TrainConfig(batch_size=512, patience=10)
train_loader = make_loader(X_tr, y_tr, batch_size=cfg.batch_size, shuffle=True)
val_loader   = make_loader(X_va, y_va, batch_size=cfg.batch_size, shuffle=False)
test_loader  = make_loader(X_te, y_te, batch_size=cfg.batch_size, shuffle=False)

weights = compute_class_weights(y_tr, n_classes=2)
model = EEGWaveNet(n_channels=21, n_classes=2)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
weights = weights.to(cfg.device)
criterion = nn.CrossEntropyLoss(weight=weights)
trainer = Trainer(model, optimizer=optimizer, criterion=criterion, config=cfg)
trainer.fit(train_loader, val_loader)
print("test:", trainer.evaluate(test_loader))

epoch   1 | train 0.5637 | val 0.4989 | val_f1b 0.0625 | sens 1.0000  *best
epoch   2 | train 0.3093 | val 0.2268 | val_f1b 0.3077 | sens 1.0000  *best
epoch   3 | train 0.1175 | val 0.0489 | val_f1b 0.8000 | sens 1.0000  *best
epoch   4 | train 0.0253 | val 0.1913 | val_f1b 1.0000 | sens 1.0000
epoch   5 | train 0.0243 | val 0.0610 | val_f1b 0.5000 | sens 1.0000
epoch   6 | train 0.0080 | val 0.0698 | val_f1b 1.0000 | sens 1.0000
epoch   7 | train 0.0038 | val 0.0160 | val_f1b 0.8000 | sens 1.0000  *best
epoch   8 | train 0.0028 | val 0.0035 | val_f1b 1.0000 | sens 1.0000  *best
epoch   9 | train 0.0009 | val 0.0162 | val_f1b 1.0000 | sens 1.0000
epoch  10 | train 0.0012 | val 0.0052 | val_f1b 1.0000 | sens 1.0000
epoch  11 | train 0.0007 | val 0.0016 | val_f1b 1.0000 | sens 1.0000  *best
epoch  12 | train 0.0005 | val 0.0011 | val_f1b 1.0000 | sens 1.0000  *best
epoch  13 | train 0.0004 | val 0.0022 | val_f1b 1.0000 | sens 1.0000
epoch  14 | train 0.0003 | val 0.0035 | val_f1b 1.0000

In [24]:
"""
Subject-dependent EEGWaveNet: train one model per CHB-MIT patient, collect
metrics into a DataFrame, and save to CSV.

Assumes these are importable (adjust the imports to match your layout):
    load_patient_data, segments_to_windows   -> data prep
    make_loader, compute_class_weights,
    Trainer, TrainConfig                      -> train.py
    EEGWaveNet                                -> eegwavenet.py

This mirrors the single-patient recipe you already use (no balancing; class
imbalance handled by class-weighted loss), just made robust enough to run
unattended across every patient.
"""

import gc
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split


# --------------------------------------------------------------------------- #
# Settings
# --------------------------------------------------------------------------- #
PATIENTS = [f"chb{n:02d}" for n in range(1, 25)]   # chb01 .. chb24

WINDOW_SAMPLES = 256 * 4        # 4 s @ 256 Hz
OVERLAP_SEC    = 0.0            # no overlap (your current choice)
TEST_FRAC      = 0.15
VAL_FRAC       = 0.15           # of the full set (taken from the train/val remainder)
MIN_PER_CLASS  = 15            # skip a patient if either class has fewer windows
SEED           = 0
RESULTS_CSV    = "results_subject_dependent.csv"

METRIC_KEYS = ["accuracy", "sensitivity", "specificity", "f1_binary", "f1_weighted"]


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_patient(patient: str, cfg: TrainConfig) -> dict:
    """Run the full pipeline for one patient. Returns a result row (dict).
    Never raises: failures/skips are reported in the 'status' field."""
    row = {"patient": patient, "status": "ok",
           "n_windows": 0, "n_pos": 0, "n_neg": 0, "pos_frac": np.nan,
           "epochs": np.nan, **{k: np.nan for k in METRIC_KEYS}}

    try:
        set_seed(SEED)

        # --- load + window ---
        X_seg, y_seg = load_patient_data(patient, normal_from_seizure_only=True)
        X, y = segments_to_windows(
            X_seg, y_seg,
            window_samples=WINDOW_SAMPLES,
            overlap_sec=OVERLAP_SEC,
            overlap_seizure_only=True,
        )

        n_pos = int((y == 1).sum())
        n_neg = int((y == 0).sum())
        row.update(n_windows=len(y), n_pos=n_pos, n_neg=n_neg,
                   pos_frac=float((y == 1).mean()))

        # --- guard: enough of each class to split stratified into 3 parts ---
        if n_pos < MIN_PER_CLASS or n_neg < MIN_PER_CLASS:
            row["status"] = f"skipped: too few windows (pos={n_pos}, neg={n_neg})"
            return row

        # --- stratified split: test, then val from the remainder ---
        X_trval, X_te, y_trval, y_te = train_test_split(
            X, y, test_size=TEST_FRAC, stratify=y, random_state=SEED
        )
        # val fraction expressed relative to the remaining train/val pool
        val_rel = VAL_FRAC / (1.0 - TEST_FRAC)
        X_tr, X_va, y_tr, y_va = train_test_split(
            X_trval, y_trval, test_size=val_rel, stratify=y_trval, random_state=SEED
        )

        # --- loaders ---
        train_loader = make_loader(X_tr, y_tr, batch_size=cfg.batch_size, shuffle=True)
        val_loader   = make_loader(X_va, y_va, batch_size=cfg.batch_size, shuffle=False)
        test_loader  = make_loader(X_te, y_te, batch_size=cfg.batch_size, shuffle=False)

        # --- model / optimizer / loss (your exact recipe) ---
        weights = compute_class_weights(y_tr, n_classes=2).to(cfg.device)
        model = EEGWaveNet(n_channels=21, n_classes=2)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss(weight=weights)
        trainer = Trainer(model, optimizer=optimizer, criterion=criterion, config=cfg)

        # --- fit + evaluate ---
        history = trainer.fit(train_loader, val_loader)
        try:
            row["epochs"] = len(history)
        except TypeError:
            pass
        metrics = trainer.evaluate(test_loader)
        row.update({k: float(metrics[k]) for k in METRIC_KEYS})

        # --- free GPU memory before the next patient ---
        del model, trainer, optimizer, criterion
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:                       # noqa: BLE001 - report, don't crash
        row["status"] = f"error: {type(exc).__name__}: {exc}"

    return row


def main():
    cfg = TrainConfig(batch_size=512, patience=10)
    rows = []
    for i, patient in enumerate(PATIENTS, 1):
        print(f"\n=== [{i}/{len(PATIENTS)}] {patient} ===")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            row = train_one_patient(patient, cfg)
        rows.append(row)
        if row["status"] == "ok":
            print(f"{patient}: acc={row['accuracy']:.3f} sens={row['sensitivity']:.3f} "
                  f"spec={row['specificity']:.3f} f1b={row['f1_binary']:.3f} "
                  f"f1w={row['f1_weighted']:.3f}")
        else:
            print(f"{patient}: {row['status']}")

    df = pd.DataFrame(rows)

    # Append a mean +/- std summary over successfully-trained patients.
    ok = df[df["status"] == "ok"]
    if len(ok):
        summary = {"patient": "MEAN", "status": f"n={len(ok)}"}
        summary.update({k: ok[k].mean() for k in METRIC_KEYS})
        std = {"patient": "STD", "status": ""}
        std.update({k: ok[k].std() for k in METRIC_KEYS})
        df = pd.concat([df, pd.DataFrame([summary, std])], ignore_index=True)

    df.to_csv(RESULTS_CSV, index=False)
    print(f"\nSaved {len(rows)} patient rows to {RESULTS_CSV}")
    print(df[["patient", "status", *METRIC_KEYS]].to_string(index=False))
    return df


if __name__ == "__main__":
    df = main()


=== [1/24] chb01 ===
epoch   1 | train 0.5387 | val 0.5360 | val_f1b 0.0399 | sens 1.0000  *best
epoch   2 | train 0.3599 | val 0.3841 | val_f1b 0.1208 | sens 1.0000  *best
epoch   3 | train 0.1936 | val 0.1854 | val_f1b 0.6818 | sens 0.9375  *best
epoch   4 | train 0.0826 | val 0.0931 | val_f1b 0.8333 | sens 0.9375  *best
epoch   5 | train 0.0352 | val 0.0497 | val_f1b 0.8205 | sens 1.0000  *best
epoch   6 | train 0.0136 | val 0.2788 | val_f1b 0.8966 | sens 0.8125
epoch   7 | train 0.0073 | val 0.1675 | val_f1b 0.8750 | sens 0.8750
epoch   8 | train 0.0032 | val 0.1396 | val_f1b 0.8750 | sens 0.8750
epoch   9 | train 0.0022 | val 0.3906 | val_f1b 0.8966 | sens 0.8125
epoch  10 | train 0.0042 | val 0.2451 | val_f1b 0.8387 | sens 0.8125
epoch  11 | train 0.0022 | val 0.0742 | val_f1b 0.8000 | sens 0.8750
epoch  12 | train 0.0020 | val 0.2428 | val_f1b 0.8966 | sens 0.8125
epoch  13 | train 0.0008 | val 0.4482 | val_f1b 0.8966 | sens 0.8125
epoch  14 | train 0.0006 | val 0.5796 | val_f1

In [25]:
df

,patient,status,n_windows,n_pos,n_neg,pos_frac,epochs,accuracy,sensitivity,specificity,f1_binary,f1_weighted
0,chb01,ok,5974.0,108.0,5866.0,0.018078,15.0,0.989967,1.000000,0.989784,0.780488,0.991042
1,chb02,ok,2036.0,42.0,1994.0,0.020629,34.0,0.996732,0.833333,1.000000,0.909091,0.996586
2,chb03,ok,6293.0,99.0,6194.0,0.015732,17.0,0.973517,1.000000,0.973089,0.545455,0.979355
3,chb04,ok,9585.0,93.0,9492.0,0.009703,19.0,0.995828,1.000000,0.995787,0.823529,0.996191
4,chb05,ok,4495.0,138.0,4357.0,0.030701,28.0,1.000000,1.000000,1.000000,1.000000,1.000000
5,chb06,ok,23295.0,36.0,23259.0,0.001545,13.0,0.877253,0.800000,0.877364,0.018307,0.933223
6,chb07,ok,8132.0,80.0,8052.0,0.009838,17.0,0.981148,1.000000,0.980960,0.510638,0.985670
7,chb08,ok,4494.0,228.0,4266.0,0.050734,14.0,0.948148,0.911765,0.950078,0.639175,0.955299
8,chb09,ok,8619.0,67.0,8552.0,0.007774,17.0,0.999227,0.900000,1.000000,0.947368,0.999206
9,chb10,ok,12608.0,109.0,12499.0,0.008645,50.0,1.000000,1.000000,1.000000,1.000000,1.000000


In [26]:
"""
Subject-independent (patient-independent) EEGWaveNet with 5 group folds.

Grouping unit = patient. A fold's test set is the windows of a held-out subset
of patients; those patients appear in NO training or validation data, so the
model is always evaluated on unseen subjects. The validation split used for
early stopping is ALSO patient-disjoint from training.

Reporting follows the paper's style (Section III-E-2): metrics are computed
per held-out test patient, then averaged across patients. Each patient is
held out exactly once (the test folds partition the patients).

Note vs. the paper: the paper uses leave-one-subject-out + an inner 5-fold,
averaged over many trials. Five group folds is a coarser, much faster
approximation of the same subject-independent idea — comparable in spirit,
not identical in protocol.

Imports (adjust to your layout):
    load_patient_data, segments_to_windows
    make_loader, compute_class_weights, Trainer, TrainConfig
    EEGWaveNet
"""

import gc
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
try:
    from sklearn.model_selection import StratifiedGroupKFold
    _HAS_SGKF = True
except ImportError:                       # older sklearn
    _HAS_SGKF = False

# --------------------------------------------------------------------------- #
# Settings
# --------------------------------------------------------------------------- #
PATIENTS       = [f"chb{n:02d}" for n in range(1, 25)]
WINDOW_SAMPLES = 256 * 4
OVERLAP_SEC    = 0.0
N_SPLITS       = 5
VAL_FRAC       = 0.15          # fraction of TRAIN patients held out for early stopping
STRATIFIED     = True          # StratifiedGroupKFold if available, else GroupKFold
SEED           = 0
RESULTS_CSV    = "results_subject_independent.csv"

METRIC_KEYS = ["accuracy", "sensitivity", "specificity", "f1_binary", "f1_weighted"]


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --------------------------------------------------------------------------- #
# Load every patient once and concatenate, tracking patient id per window.
# Stored as float32 to keep the combined array manageable in RAM.
# --------------------------------------------------------------------------- #
def load_all(patients):
    Xs, ys, groups = [], [], []
    for p in patients:
        try:
            X_seg, y_seg = load_patient_data(p, normal_from_seizure_only=True)
            X, y = segments_to_windows(
                X_seg, y_seg,
                window_samples=WINDOW_SAMPLES,
                overlap_sec=OVERLAP_SEC,
                overlap_seizure_only=True,
            )
        except Exception as exc:                       # noqa: BLE001
            print(f"  skip {p}: {type(exc).__name__}: {exc}")
            continue
        Xs.append(X.astype(np.float32))
        ys.append(y.astype(np.int64))
        groups.append(np.full(len(y), p))
        print(f"  {p}: {len(y):5d} windows  (pos={int((y == 1).sum())})")

    if not Xs:
        raise ValueError("No patient data loaded.")
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0)
    groups = np.concatenate(groups, axis=0)
    print(f"Total: {len(y)} windows from {len(np.unique(groups))} patients "
          f"({X.nbytes / 1e9:.2f} GB in float32)")
    return X, y, groups


def make_splitter():
    if STRATIFIED and _HAS_SGKF:
        return StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    return GroupKFold(n_splits=N_SPLITS)        # deterministic, no shuffle/stratify


def main():
    set_seed(SEED)
    cfg = TrainConfig(batch_size=512, patience=10)

    print("Loading all patients...")
    X, y, groups = load_all(PATIENTS)

    splitter = make_splitter()
    rows = []

    for fold, (trval_idx, te_idx) in enumerate(splitter.split(X, y, groups), 1):
        test_patients = sorted(np.unique(groups[te_idx]).tolist())
        print(f"\n=== Fold {fold}/{N_SPLITS} | test patients: {test_patients} ===")

        # --- patient-disjoint validation split from the training pool ---
        gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
        tr_rel, va_rel = next(gss.split(X[trval_idx], y[trval_idx], groups[trval_idx]))
        tr_idx = trval_idx[tr_rel]
        va_idx = trval_idx[va_rel]

        # --- loaders ---
        train_loader = make_loader(X[tr_idx], y[tr_idx], batch_size=cfg.batch_size, shuffle=True)
        val_loader   = make_loader(X[va_idx], y[va_idx], batch_size=cfg.batch_size, shuffle=False)

        # --- model / optimizer / loss ---
        set_seed(SEED)                          # comparable init across folds
        weights = compute_class_weights(y[tr_idx], n_classes=2).to(cfg.device)
        model = EEGWaveNet(n_channels=21, n_classes=2)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss(weight=weights)
        trainer = Trainer(model, optimizer=optimizer, criterion=criterion, config=cfg)

        history = trainer.fit(train_loader, val_loader)
        epochs = len(history) if hasattr(history, "__len__") else np.nan

        # --- evaluate PER held-out patient (paper-style aggregation) ---
        for p in test_patients:
            mask = groups[te_idx] == p
            idx = te_idx[mask]
            loader = make_loader(X[idx], y[idx], batch_size=cfg.batch_size, shuffle=False)
            m = trainer.evaluate(loader)
            rows.append({
                "patient": p, "fold": fold, "status": "ok",
                "n_windows": int(mask.sum()),
                "n_pos": int((y[idx] == 1).sum()),
                "n_neg": int((y[idx] == 0).sum()),
                "epochs": epochs,
                **{k: float(m[k]) for k in METRIC_KEYS},
            })
            print(f"  {p}: acc={m['accuracy']:.3f} sens={m['sensitivity']:.3f} "
                  f"spec={m['specificity']:.3f} f1b={m['f1_binary']:.3f}")

        # --- free GPU memory before next fold ---
        del model, trainer, optimizer, criterion
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame(rows).sort_values("patient").reset_index(drop=True)

    # mean +/- std across patients (the paper's headline aggregation)
    summary = {"patient": "MEAN", "fold": "", "status": f"n={len(df)}"}
    summary.update({k: df[k].mean() for k in METRIC_KEYS})
    std = {"patient": "STD", "fold": "", "status": ""}
    std.update({k: df[k].std() for k in METRIC_KEYS})
    df = pd.concat([df, pd.DataFrame([summary, std])], ignore_index=True)

    df.to_csv(RESULTS_CSV, index=False)
    print(f"\nSaved to {RESULTS_CSV}")
    print(df[["patient", "fold", *METRIC_KEYS]].to_string(index=False))
    return df


if __name__ == "__main__":
    df = main()

Loading all patients...
  chb01:  5974 windows  (pos=108)
  chb02:  2036 windows  (pos=42)
  chb03:  6293 windows  (pos=99)
  chb04:  9585 windows  (pos=93)
  chb05:  4495 windows  (pos=138)
  chb06: 23295 windows  (pos=36)
  chb07:  8132 windows  (pos=80)
  chb08:  4494 windows  (pos=228)
  chb09:  8619 windows  (pos=67)
  chb10: 12608 windows  (pos=109)
  chb11:  2512 windows  (pos=201)


/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=T

  chb12:  8694 windows  (pos=238)


/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, 

  chb13:  6289 windows  (pos=107)


...

  chb18:  5062 windows  (pos=77)


/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, 

  chb19:  2632 windows  (pos=58)
  chb20:  4999 windows  (pos=69)


/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=T

  chb21:  3444 windows  (pos=49)
  chb22:  2695 windows  (pos=50)
  chb23:  8056 windows  (pos=103)
  chb24: 10784 windows  (pos=122)
Total: 166773 windows from 24 patients (14.35 GB in float32)

=== Fold 1/5 | test patients: ['chb02', 'chb13', 'chb14', 'chb15', 'chb16', 'chb22'] ===
epoch   1 | train 0.3561 | val 0.5072 | val_f1b 0.1796 | sens 0.8509  *best
epoch   2 | train 0.1944 | val 0.4004 | val_f1b 0.3261 | sens 0.6055  *best
epoch   3 | train 0.1443 | val 0.4295 | val_f1b 0.2792 | sens 0.7546
epoch   4 | train 0.1113 | val 0.5086 | val_f1b 0.2789 | sens 0.6858
epoch   5 | train 0.0895 | val 0.4563 | val_f1b 0.3451 | sens 0.5963
epoch   6 | train 0.0719 | val 0.5288 | val_f1b 0.3061 | sens 0.6491
epoch   7 | train 0.0553 | val 0.5237 | val_f1b 0.3765 | sens 0.5803
epoch   8 | train 0.0412 | val 0.4641 | val_f1b 0.3055 | sens 0.7454
epoch   9 | train 0.0401 | val 0.5258 | val_f1b 0.3779 | sens 0.6812
epoch  10 | train 0.0385 | val 0.6075 | val_f1b 0.4263 | sens 0.6101
epoch  11 |

In [27]:
df

,patient,fold,status,n_windows,n_pos,n_neg,epochs,accuracy,sensitivity,specificity,f1_binary,f1_weighted
0,chb01,3,ok,5974.0,108.0,5866.0,12.0,0.866254,0.990741,0.863962,0.211254,0.913994
1,chb02,1,ok,2036.0,42.0,1994.0,12.0,0.981827,0.571429,0.990471,0.564706,0.981932
2,chb03,4,ok,6293.0,99.0,6194.0,11.0,0.939774,0.838384,0.941395,0.304587,0.958079
3,chb04,4,ok,9585.0,93.0,9492.0,11.0,0.988106,0.645161,0.991466,0.512821,0.989311
4,chb05,5,ok,4495.0,138.0,4357.0,11.0,0.053838,0.992754,0.024099,0.060526,0.047467
5,chb06,5,ok,23295.0,36.0,23259.0,11.0,0.323889,0.888889,0.323015,0.004047,0.487489
6,chb07,5,ok,8132.0,80.0,8052.0,11.0,0.784063,0.937500,0.782538,0.078699,0.869839
7,chb08,3,ok,4494.0,228.0,4266.0,12.0,0.719181,0.750000,0.717534,0.213217,0.797844
8,chb09,5,ok,8619.0,67.0,8552.0,11.0,0.142244,0.985075,0.135641,0.017542,0.237135
9,chb10,5,ok,12608.0,109.0,12499.0,11.0,0.187183,1.000000,0.180094,0.020829,0.302762


In [35]:
import numpy as np
import pandas as pd

PATIENTS = [f"chb{n:02d}" for n in range(1, 25)]

# ---- per-window features: per-channel log-variance (21-dim) ----------------
# Reflects the uV-scaled amplitude distribution the network actually sees.
# Swap in band-powers here if you want spectral sensitivity too.
def window_features(X):                       # X: (n, 21, 1024) -> (n, 21)
    return np.log(X.var(axis=2) + 1e-12)

# ---- RBF-MMD^2 (unbiased) with a FIXED shared gamma for comparability ------
def _sqdists(A, B):
    a2 = (A ** 2).sum(1)[:, None]
    b2 = (B ** 2).sum(1)[None, :]
    return np.maximum(a2 + b2 - 2 * A @ B.T, 0.0)

def rbf_mmd2(X, Y, gamma, max_samples=400, seed=0):
    rng = np.random.default_rng(seed)
    if len(X) > max_samples: X = X[rng.choice(len(X), max_samples, replace=False)]
    if len(Y) > max_samples: Y = Y[rng.choice(len(Y), max_samples, replace=False)]
    m, n = len(X), len(Y)
    if m < 2 or n < 2: return np.nan
    Kxx = np.exp(-gamma * _sqdists(X, X))
    Kyy = np.exp(-gamma * _sqdists(Y, Y))
    Kxy = np.exp(-gamma * _sqdists(X, Y))
    sxx = (Kxx.sum() - np.trace(Kxx)) / (m * (m - 1))
    syy = (Kyy.sum() - np.trace(Kyy)) / (n * (n - 1))
    sxy = Kxy.mean()
    return float(sxx + syy - 2 * sxy)         # can be slightly <0 (unbiased)

# ---- build per-patient pools of normal / seizure features ------------------
pools = {}                                    # patient -> {'normal':(.,21),'seizure':(.,21)}
for p in PATIENTS:
    try:
        X_seg, y_seg = load_patient_data(p, normal_from_seizure_only=True)
        X, y = segments_to_windows(X_seg, y_seg, window_samples=256 * 4,
                                   overlap_sec=0.0, overlap_seizure_only=True)
    except Exception as exc:
        print(f"skip {p}: {type(exc).__name__}: {exc}")
        continue
    f = window_features(X)
    pools[p] = {"normal": f[y == 0], "seizure": f[y == 1]}

# ---- standardize features globally, then fix gamma via the median heuristic -
allf = np.concatenate([pools[p][c] for p in pools for c in ("normal", "seizure")
                       if len(pools[p][c])])
mu, sd = allf.mean(0), allf.std(0) + 1e-8
for p in pools:
    for c in ("normal", "seizure"):
        if len(pools[p][c]):
            pools[p][c] = (pools[p][c] - mu) / sd

rng = np.random.default_rng(0)
allf_std = (allf - mu) / sd
samp = allf_std[rng.choice(len(allf_std), min(1500, len(allf_std)), replace=False)]
med_sq = np.median(_sqdists(samp, samp))
GAMMA = 1.0 / (med_sq + 1e-12)                # shared across every pair

# ---- pairwise MMD^2 matrix per class ---------------------------------------
def mmd_matrix(cls):
    pats = [p for p in pools if len(pools[p][cls]) >= 2]
    M = pd.DataFrame(np.nan, index=pats, columns=pats, dtype=float)
    for i, a in enumerate(pats):
        for b in pats[i + 1:]:
            v = rbf_mmd2(pools[a][cls], pools[b][cls], GAMMA)
            M.loc[a, b] = M.loc[b, a] = v
    return M

mmd_normal  = mmd_matrix("normal")
mmd_seizure = mmd_matrix("seizure")

def offdiag_mean(M):
    vals = M.values[~np.eye(len(M), dtype=bool)]
    return np.nanmean(vals)

print(f"mean inter-patient MMD^2  normal : {offdiag_mean(mmd_normal):.4f}")
print(f"mean inter-patient MMD^2  seizure: {offdiag_mean(mmd_seizure):.4f}")
# per-patient 'how different is this patient from the rest' (mean over its row)
summary = pd.DataFrame({
    "normal_MMD":  mmd_normal.mean(axis=1, skipna=True),
    "seizure_MMD": mmd_seizure.mean(axis=1, skipna=True),
}).sort_values("seizure_MMD", ascending=False)
print(summary.round(4))

/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=True, verbose=False)
/tmp/ipykernel_50882/1950713380.py:68: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(str(filepath), preload=T

mean inter-patient MMD^2  normal : 0.3808
mean inter-patient MMD^2  seizure: 0.5573
       normal_MMD  seizure_MMD
chb09      0.3337       0.9925
chb15      0.5586       0.9614
chb12      0.3438       0.7282
chb05      0.4382       0.6501
chb21      0.3371       0.6293
chb17      0.3605       0.5973
chb14      0.5034       0.5810
chb06      0.6269       0.5589
chb10      0.6017       0.5587
chb23      0.3573       0.5508
chb11      0.2881       0.5384
chb24      0.2550       0.5364
chb22      0.2765       0.5274
chb01      0.2932       0.5146
chb02      0.2613       0.5069
chb08      0.3839       0.4978
chb20      0.4866       0.4793
chb07      0.2712       0.4767
chb18      0.4570       0.4731
chb13      0.3582       0.4485
chb16      0.2820       0.4443
chb03      0.3540       0.4116
chb19      0.2769       0.3881
chb04      0.4353       0.3232


In [36]:
mmd_seizure

,chb01,chb02,chb03,chb04,chb05,chb06,chb07,chb08,chb09,chb10,...,chb15,chb16,chb17,chb18,chb19,chb20,chb21,chb22,chb23,chb24
chb01,NaN,0.176010,0.171196,0.340734,0.478039,0.457581,0.251507,0.120750,0.950026,0.305033,...,1.320231,0.432514,0.727106,0.525475,0.296508,0.533545,0.829947,0.231115,0.516977,0.440919
chb02,0.176010,NaN,0.270432,0.318609,0.460019,0.366365,0.264878,0.127641,0.884793,0.263184,...,1.219146,0.545137,0.758747,0.597493,0.270984,0.457696,0.819679,0.245961,0.607437,0.478229
chb03,0.171196,0.270432,NaN,0.259288,0.553792,0.362315,0.286978,0.265102,0.989540,0.360486,...,1.054756,0.259056,0.460653,0.331640,0.275898,0.384508,0.481272,0.191892,0.190803,0.425200
chb04,0.340734,0.318609,0.259288,NaN,0.414050,0.297858,0.270241,0.310472,0.697606,0.372950,...,0.585672,0.185454,0.303170,0.174319,0.136489,0.192634,0.348876,0.378634,0.399075,0.319638
chb05,0.478039,0.460019,0.553792,0.414050,NaN,0.910456,0.079235,0.231964,0.124027,0.210308,...,1.227245,0.827519,1.059171,0.886422,0.113289,0.866274,1.139530,0.640524,0.942300,0.175342
chb06,0.457581,0.366365,0.362315,0.297858,0.910456,NaN,0.651064,0.577004,1.404326,0.712472,...,1.101270,0.338946,0.502484,0.414006,0.568180,0.113174,0.563120,0.434269,0.506213,0.757007
chb07,0.251507,0.264878,0.286978,0.270241,0.079235,0.651064,NaN,0.123106,0.311429,0.109396,...,1.091891,0.571292,0.816173,0.633711,0.061323,0.633055,0.871242,0.395521,0.655213,0.122121
chb08,0.120750,0.127641,0.265102,0.310472,0.231964,0.577004,0.123106,NaN,0.577599,0.136029,...,1.238312,0.576221,0.824853,0.655301,0.136111,0.608068,0.925614,0.264034,0.636035,0.290380
chb09,0.950026,0.884793,0.989540,0.697606,0.124027,1.404326,0.311429,0.577599,NaN,0.496963,...,1.459187,1.221952,1.432820,1.267435,0.309038,1.283835,1.499209,1.095199,1.353863,0.412513
chb10,0.305033,0.263184,0.360486,0.372950,0.210308,0.712472,0.109396,0.136029,0.496963,NaN,...,1.241718,0.711280,0.954095,0.799749,0.157022,0.735195,1.028798,0.281977,0.699905,0.256402


In [37]:
mmd_normal

,chb01,chb02,chb03,chb04,chb05,chb06,chb07,chb08,chb09,chb10,...,chb15,chb16,chb17,chb18,chb19,chb20,chb21,chb22,chb23,chb24
chb01,NaN,0.201867,0.356507,0.359192,0.378079,0.588720,0.217537,0.349997,0.227024,0.577655,...,0.520481,0.265813,0.330389,0.407358,0.107517,0.109883,0.270409,0.210167,0.277074,0.078470
chb02,0.201867,NaN,0.322551,0.330311,0.229849,0.349642,0.191329,0.144793,0.128024,0.364222,...,0.543457,0.170497,0.324945,0.429759,0.217682,0.407111,0.240876,0.126019,0.279666,0.127781
chb03,0.356507,0.322551,NaN,0.100953,0.666806,0.774507,0.106674,0.570593,0.520041,0.802200,...,0.096537,0.191348,0.090024,0.072598,0.129767,0.472906,0.187834,0.272793,0.226278,0.300130
chb04,0.359192,0.330311,0.100953,NaN,0.746108,0.768745,0.231646,0.655003,0.626401,0.850567,...,0.097847,0.323658,0.204736,0.065824,0.212761,0.531594,0.372666,0.441694,0.408094,0.400700
chb05,0.378079,0.229849,0.666806,0.746108,NaN,0.224623,0.511798,0.110604,0.086596,0.133103,...,0.948146,0.398808,0.672906,0.798493,0.507259,0.641828,0.589119,0.342654,0.601733,0.281214
chb06,0.588720,0.349642,0.774507,0.768745,0.224623,NaN,0.727758,0.329223,0.307159,0.116268,...,1.078227,0.589090,0.885149,0.930629,0.736598,0.917627,0.878477,0.636839,0.901416,0.569903
chb07,0.217537,0.191329,0.106674,0.231646,0.511798,0.727758,NaN,0.406902,0.333908,0.656592,...,0.239475,0.084958,0.092031,0.172223,0.080263,0.403784,0.047084,0.101270,0.108032,0.159848
chb08,0.349997,0.144793,0.570593,0.655003,0.110604,0.329223,0.406902,NaN,0.070539,0.284858,...,0.844286,0.290664,0.539702,0.711383,0.441411,0.598919,0.461605,0.202087,0.477895,0.230139
chb09,0.227024,0.128024,0.520041,0.626401,0.086596,0.307159,0.333908,0.070539,NaN,0.211634,...,0.809484,0.254885,0.510033,0.672601,0.364522,0.500372,0.403540,0.190637,0.424196,0.168663
chb10,0.577655,0.364222,0.802200,0.850567,0.133103,0.116268,0.656592,0.284858,0.211634,NaN,...,1.095976,0.543056,0.891011,0.949922,0.706320,0.926058,0.837328,0.572228,0.881806,0.474104
